In [1]:
# ============================================================
# M1 — LOAD FINAL SCALED EI01 NETWORK (EXPORT HUB, FULL YEAR)
# ELECTRICITY EXPORT ONLY 
# ============================================================

import pypsa
import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# PATHS (HARD, EXPLICIT — NO KERNEL STATE)
# ------------------------------------------------------------
root = Path(r"C:\Users\franc\pypsa\pypsa-eur")

SCENARIO = "EI01_export_hub"

base_fn = (
    root
    / "results"
    / "networks"
    / f"{SCENARIO}.nc"
)

print("M1 loading network from:")
print(base_fn)

assert base_fn.exists(), f"Scaled EI01 network not found: {base_fn}"

# ------------------------------------------------------------
# LOAD NETWORK (FULL YEAR, 8760)
# ------------------------------------------------------------
n = pypsa.Network(base_fn)
print("Loaded:", base_fn.name)
print("Snapshots:", len(n.snapshots))

# ------------------------------------------------------------
# SAFETY — ENSURE CARRIER TABLE IS NETCDF / PYPSA SAFE
# ------------------------------------------------------------
if "color" in n.carriers.columns:
    n.carriers["color"] = n.carriers["color"].astype(str)

# Ensure all generator carriers exist in n.carriers
n.generators["carrier"] = n.generators["carrier"].astype(str)
missing_carriers = set(n.generators.carrier.unique()) - set(n.carriers.index)

if missing_carriers:
    print("INFO: Adding missing carriers to n.carriers:", sorted(missing_carriers))
    default_row = {c: 0 for c in n.carriers.columns}
    for mc in sorted(missing_carriers):
        n.carriers.loc[mc] = default_row

# ------------------------------------------------------------
# DISPATCH-ONLY (ABSOLUTE — NO EXPANSION)
# ------------------------------------------------------------
for df_name in ["generators", "links", "lines", "storage_units"]:
    if hasattr(n, df_name):
        df = getattr(n, df_name)
        if "p_nom_extendable" in df.columns:
            df["p_nom_extendable"] = False

if hasattr(n, "lines") and "s_nom_extendable" in n.lines.columns:
    n.lines["s_nom_extendable"] = False

print("✓ Expansion disabled (dispatch-only mode)")

# ------------------------------------------------------------
# STORAGE — NON-CYCLIC, DAILY ROLLING COMPATIBLE
# (Battery only, spike smoothing)
# ------------------------------------------------------------
if hasattr(n, "storage_units") and not n.storage_units.empty:
    if "cyclic_state_of_charge" in n.storage_units.columns:
        n.storage_units["cyclic_state_of_charge"] = False
    if "state_of_charge_initial" not in n.storage_units.columns:
        n.storage_units["state_of_charge_initial"] = 0.0

print("✓ Battery storage set to non-cyclic with explicit initial SOC")

# ------------------------------------------------------------
# LOAD SHEDDING — LAST RESORT (GLOBAL SAFETY)
# ------------------------------------------------------------
LS_COST = 10_000.0  # €/MWh

if "load_shedding" not in n.carriers.index:
    n.carriers.loc["load_shedding"] = {c: 0 for c in n.carriers.columns}

missing_ls = [
    b for b in n.buses.index
    if f"load_shed_{b}" not in n.generators.index
]

for b in missing_ls:
    n.add(
        "Generator",
        name=f"load_shed_{b}",
        bus=b,
        carrier="load_shedding",
        p_nom=1e6,
        p_min_pu=0.0,
        p_max_pu=1.0,
        marginal_cost=LS_COST,
    )

print("✓ Load shedding generators ensured:", len(missing_ls))

# ------------------------------------------------------------
# SNAPSHOT BACKUP (FOR M2 / M3 DAILY SLICING)
# ------------------------------------------------------------
FULL_SNAPSHOTS = n.snapshots.copy()

# ------------------------------------------------------------
# OUTPUT DIRECTORIES (EI01-SPECIFIC)
# ------------------------------------------------------------
out_root = root / "results" / "daily_runs_EI01"
by_day   = out_root / "by_day"
by_month = out_root / "by_month"

by_day.mkdir(parents=True, exist_ok=True)
by_month.mkdir(parents=True, exist_ok=True)

print("✓ Output directories ready:")
print(" ", out_root)

# ------------------------------------------------------------
# SOLVER
# ------------------------------------------------------------
SOLVER = "cbc"
print("Solver set to:", SOLVER)

# ------------------------------------------------------------
# SANITY CHECK — EXPORT HUB CORE COMPONENTS
# ------------------------------------------------------------
assert any(n.generators.carrier.str.contains("offwind")), "Offshore wind missing"
assert (n.links.carrier == "DC").any(), "HVDC export links missing"
assert not (n.links.carrier == "electrolysis").any(), "Electrolysis should NOT exist"
assert not (n.buses.carrier == "hydrogen").any(), "Hydrogen bus should NOT exist"

print("✓ Export Hub core components detected (electric-only)")

print("\nM1_EI01 READY — proceed to M2")


M1 loading network from:
C:\Users\franc\pypsa\pypsa-eur\results\networks\EI01_export_hub.nc


INFO:pypsa.io:Imported network EI01_export_hub.nc has buses, carriers, generators, lines, links, loads, storage_units, stores


Loaded: EI01_export_hub.nc
Snapshots: 8760
✓ Expansion disabled (dispatch-only mode)
✓ Battery storage set to non-cyclic with explicit initial SOC
✓ Load shedding generators ensured: 0
✓ Output directories ready:
  C:\Users\franc\pypsa\pypsa-eur\results\daily_runs_EI01
Solver set to: cbc
✓ Export Hub core components detected (electric-only)

M1_EI01 READY — proceed to M2


In [2]:
# ============================================================
# M2_EI01 — Helpers for one-day solve, metrics, and rolling states
# EXPORT HUB SCENARIO (ELECTRIC-ONLY)
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# CONSTANTS
# ------------------------------------------------------------

EI_BUS = "EI01_bus"
EI_COUNTRY = "EI"

MAIN_COUNTRIES = ["DK", "DE", "NL", "NO", "SE"]

# ------------------------------------------------------------
# Country mappings (EXCLUDING Energy Island)
# ------------------------------------------------------------

bus_country  = n.buses.country
gen_country  = n.generators.bus.map(bus_country)
load_country = n.loads.bus.map(bus_country)

# ------------------------------------------------------------
# Defensive: ensure all generator carriers exist in n.carriers
# ------------------------------------------------------------

n.generators["carrier"] = n.generators["carrier"].astype(str)
missing_carriers = set(n.generators.carrier.unique()) - set(n.carriers.index)

if missing_carriers:
    print("INFO (M2_EI01): adding missing carriers:", sorted(missing_carriers))
    default_row = {c: 0 for c in n.carriers.columns}
    for mc in missing_carriers:
        n.carriers.loc[mc] = default_row

# ------------------------------------------------------------
# Snapshot helpers
# ------------------------------------------------------------

def day_index(ts: pd.Timestamp) -> pd.DatetimeIndex:
    d = pd.Timestamp(ts).normalize()
    hours = pd.date_range(d, d + pd.Timedelta(hours=23), freq="h")
    return hours.intersection(FULL_SNAPSHOTS)

# ------------------------------------------------------------
# Rolling storage states (battery only)
# ------------------------------------------------------------

def roll_storage_initials_from_solution():
    if hasattr(n, "storage_units_t") and "state_of_charge" in n.storage_units_t:
        last_soc = (
            n.storage_units_t.state_of_charge.iloc[-1]
            .reindex(n.storage_units.index)
            .fillna(0.0)
        )
        if "state_of_charge_initial" not in n.storage_units.columns:
            n.storage_units["state_of_charge_initial"] = 0.0
        n.storage_units["state_of_charge_initial"] = last_soc

# ------------------------------------------------------------
# Metric extraction — EI01 LOGIC (NO H2)
# ------------------------------------------------------------

def extract_day_metrics(day_idx: pd.DatetimeIndex) -> dict:

    out = {}

    # -----------------
    # Core time series
    # -----------------

    loads_ts = n.loads_t.p_set.loc[day_idx]      # MW
    gens_ts  = n.generators_t.p.loc[day_idx]     # MW
    links_ts = n.links_t.p0.loc[day_idx] if hasattr(n, "links_t") else None

    # -----------------
    # DEMAND (EU, excluding EI)
    # -----------------

    out["demand_EU_MW"] = float(loads_ts.sum(axis=1).mean())

    for c in MAIN_COUNTRIES:
        mask = (load_country == c)
        if mask.any():
            out[f"demand_{c}_MW"] = float(
                loads_ts.loc[:, mask].sum(axis=1).mean()
            )

    # -----------------
    # ELECTRIC GENERATION (EXCL. LOAD SHEDDING)
    # -----------------

    elec_gen_mask = ~n.generators.carrier.isin(["load_shedding"])
    elec_gen_ts = gens_ts.loc[:, elec_gen_mask]

    out["generation_EU_MW"] = float(elec_gen_ts.sum(axis=1).mean())

    for c in MAIN_COUNTRIES:
        mask = (gen_country == c) & elec_gen_mask
        if mask.any():
            out[f"generation_{c}_MW"] = float(
                elec_gen_ts.loc[:, mask].sum(axis=1).mean()
            )

    # -----------------
    # ELECTRIC EXPORT FROM ISLAND (HVDC)
    # -----------------

    if links_ts is not None:
        hvdc_mask = (n.links.bus0 == EI_BUS)
        elec_export_ts = links_ts.loc[:, hvdc_mask].sum(axis=1)
        out["elec_export_MW"] = float(elec_export_ts.mean())
    else:
        out["elec_export_MW"] = 0.0

    # -----------------
    # NET ELECTRIC BALANCE (EU)
    # -----------------

    out["net_elec_balance_EU_MW"] = (
        out["generation_EU_MW"]
        - out["demand_EU_MW"]
    )

    # -----------------
    # PRICES (LOAD-WEIGHTED, EXCLUDING EI)
    # -----------------

    if hasattr(n, "buses_t") and "marginal_price" in n.buses_t:
        prices_ts = n.buses_t.marginal_price.loc[day_idx]

        out["price_EU_EUR_per_MWh"] = float(
            prices_ts.loc[:, bus_country != EI_COUNTRY].stack().mean()
        )

        for c in MAIN_COUNTRIES:
            mask = (bus_country == c)
            if mask.any():
                out[f"price_{c}_EUR_per_MWh"] = float(
                    prices_ts.loc[:, mask].mean(axis=1).mean()
                )

    # -----------------
    # LOAD SHEDDING
    # -----------------

    if (n.generators.carrier == "load_shedding").any():
        shed_ts = gens_ts.loc[:, n.generators.carrier == "load_shedding"]
        out["load_shed_MW"] = float(shed_ts.sum(axis=1).mean())
    else:
        out["load_shed_MW"] = 0.0

    return out

# ------------------------------------------------------------
# One-day solve wrapper
# ------------------------------------------------------------

def solve_one_day(
    d: pd.Timestamp,
    solver: str = None,
    threads: int = 4,
    logdir=None,
) -> dict:

    idx = day_index(d)
    if len(idx) == 0:
        return {"note": "no snapshots"}

    n.set_snapshots(idx)

    status, cond = n.optimize(
        solver_name=(solver or SOLVER),
        threads=threads,
        progress=False,
    )

    if status != "ok":
        return {"note": f"solve status {status} ({cond})"}

    metrics = extract_day_metrics(idx)
    roll_storage_initials_from_solution()
    return metrics

# ------------------------------------------------------------
# Smoke test
# ------------------------------------------------------------

d0 = pd.Timestamp("2013-01-01")
print("Testing EI01 day:", d0.date())
print(solve_one_day(d0, solver=SOLVER))
print("M2_EI01 ready.")


Testing EI01 day: 2013-01-01


INFO:linopy.model: Solve problem using Cbc solver
INFO:linopy.model:Solver options:
 - threads: 4
INFO:linopy.io: Writing time: 0.57s
INFO:linopy.solvers:Welcome to the CBC MILP Solver 
Version: 2.10.12 
Build Date: Aug  2 2025 

command line - cbc -printingOptions all -import C:\Users\franc\AppData\Local\Temp\linopy-problem-1r6zvr6l.lp -threads 4 -solve -solu C:\Users\franc\AppData\Local\Temp\linopy-solve-3xdhgz5i.sol (default strategy 1)
Option for printingOptions changed from normal to all
No match for threads - ? for list of commands
No match for 4 - ? for list of commands
Presolve 7252 (-62592) rows, 16689 (-17035) columns and 33827 (-84838) elements
Perturbing problem by 0.001% of 763.73472 - largest nonzero change 0.00086369541 ( 0.10014579%) - largest zero change 0.00086368374
0  Obj -173.78136 Primal inf 16214754 (2056)
220  Obj -173.78136 Primal inf 18329397 (2030)
440  Obj 814046.01 Primal inf 21802834 (2114)
660  Obj 980815.31 Primal inf 18307657 (2221)
880  Obj 1182622.2 P

{'demand_EU_MW': 321042.7592837016, 'demand_DK_MW': 3518.5416673024497, 'demand_DE_MW': 41525.18316332499, 'demand_NL_MW': 11271.708331425985, 'demand_NO_MW': 16243.166668256124, 'demand_SE_MW': 15982.95834604899, 'generation_EU_MW': 295167.19949497376, 'generation_DK_MW': 4426.929764554167, 'generation_DE_MW': 56162.101601666676, 'generation_NL_MW': 7942.780665416667, 'generation_NO_MW': 1400.184262075, 'generation_SE_MW': 12673.056077041665, 'elec_export_MW': 2754.707337916667, 'net_elec_balance_EU_MW': -25875.55978872784, 'price_EU_EUR_per_MWh': 8.626017168611112, 'price_DK_EUR_per_MWh': 4.978919381944444, 'price_DE_EUR_per_MWh': 6.149684638888888, 'price_NL_EUR_per_MWh': 4.520590069444444, 'price_NO_EUR_per_MWh': 5.186035708333333, 'price_SE_EUR_per_MWh': 5.186035708333333, 'load_shed_MW': 0.0}
M2_EI01 ready.


In [4]:
# ============================================================
# FINAL M3_EI01 — FULL EXPORT (DAILY + HOURLY)
# EXPORT HUB SCENARIO (ELECTRIC-ONLY)
# BASELINE-ALIGNED
# ============================================================

import pandas as pd
import numpy as np
import pypsa
from pathlib import Path

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
SCENARIO = "EI01"
months = [f"2013-{m:02d}" for m in range(1, 13)]

MAIN_COUNTRIES = ["DK", "DE", "NL", "NO", "SE", "BE", "FR", "PL", "PT"]

root = Path(r"C:\Users\franc\pypsa\pypsa-eur")
base_fn  = root / "results" / "networks" / "EI01_export_hub.nc"
out_root = root / "results" / f"daily_runs_{SCENARIO}"
out_root.mkdir(parents=True, exist_ok=True)

SOLVER = "cbc"

print("=== FINAL M3_EI01 — COUNTRY-CORRECT (EXPORT HUB) ===")
print("Network:", base_fn)
assert base_fn.exists(), f"Network not found: {base_fn}"

# ------------------------------------------------------------
# SNAPSHOT HELPER
# ------------------------------------------------------------
def day_index(ts, full_snapshots):
    d = pd.Timestamp(ts).normalize()
    hours = pd.date_range(d, d + pd.Timedelta(hours=23), freq="h")
    return hours.intersection(full_snapshots)

# ------------------------------------------------------------
# MAIN LOOP
# ------------------------------------------------------------
for month in months:
    print(f"\n=== Month {month} ===")

    month_start = pd.Timestamp(f"{month}-01")
    month_end   = month_start + pd.offsets.MonthEnd(1)
    days = pd.date_range(month_start, month_end, freq="D")

    rows_daily  = []
    rows_hourly = []

    last_soc = None
    last_e   = None

    for d in days:
        # ----------------------------------------------------
        # LOAD NETWORK FRESH
        # ----------------------------------------------------
        n = pypsa.Network(base_fn)
        FULL_SNAPSHOTS = n.snapshots.copy()

        idx = day_index(d, FULL_SNAPSHOTS)
        if len(idx) == 0:
            continue

        print(f"→ {d.date()} ({len(idx)} h)")

        # ----------------------------------------------------
        # DISPATCH ONLY
        # ----------------------------------------------------
        for comp in ["generators", "links", "lines", "storage_units", "stores"]:
            if hasattr(n, comp):
                df = getattr(n, comp)
                if "p_nom_extendable" in df.columns:
                    df["p_nom_extendable"] = False

        if "s_nom_extendable" in n.lines.columns:
            n.lines["s_nom_extendable"] = False

        # ----------------------------------------------------
        # STORAGE (ROLLING — BATTERY ONLY)
        # ----------------------------------------------------
        if not n.storage_units.empty:
            n.storage_units["cyclic_state_of_charge"] = False
            if "state_of_charge_initial" not in n.storage_units.columns:
                n.storage_units["state_of_charge_initial"] = 0.0

        if last_soc is not None and not n.storage_units.empty:
            n.storage_units["state_of_charge_initial"] = (
                last_soc.reindex(n.storage_units.index).fillna(0.0)
            )

        # ----------------------------------------------------
        # LOAD SHEDDING (SAFETY)
        # ----------------------------------------------------
        if "load_shedding" not in n.carriers.index:
            n.carriers.loc["load_shedding"] = {c: 0 for c in n.carriers.columns}

        for b in n.buses.index:
            g = f"load_shed_{b}"
            if g not in n.generators.index:
                n.add(
                    "Generator",
                    name=g,
                    bus=b,
                    carrier="load_shedding",
                    p_nom=1e6,
                    marginal_cost=10_000.0,
                )

        # ----------------------------------------------------
        # SOLVE DAY
        # ----------------------------------------------------
        n.set_snapshots(idx)
        status, cond = n.optimize(
            solver_name=SOLVER,
            threads=4,
            progress=False,
        )

        if status != "ok":
            rows_daily.append({"date": d.date(), "note": f"{status} ({cond})"})
            continue

        # ----------------------------------------------------
        # SAVE STORAGE STATE
        # ----------------------------------------------------
        if not n.storage_units.empty:
            last_soc = n.storage_units_t.state_of_charge.iloc[-1].copy()

        # ----------------------------------------------------
        # BASE TIME SERIES
        # ----------------------------------------------------
        load_p  = n.loads_t.p_set.loc[idx]
        gen_p   = n.generators_t.p.loc[idx]
        prices  = n.buses_t.marginal_price.loc[idx]

        total_demand_MW     = load_p.sum(axis=1)
        total_generation_MW = gen_p.sum(axis=1)

        hourly_df = pd.DataFrame({
            "snapshot": idx,
            "date": idx.normalize(),
            "hour": idx.hour,
            "total_demand_MW": total_demand_MW.values,
            "total_generation_MW": total_generation_MW.values,
        })

        # ----------------------------------------------------
        # COUNTRY MAPS (BASELINE-COMPATIBLE)
        # ----------------------------------------------------
        bus_country  = n.buses.country
        load_country = n.loads.bus.map(bus_country)
        gen_country  = n.generators.bus.map(bus_country)

        # ----------------------------------------------------
        # DEMAND BY COUNTRY
        # ----------------------------------------------------
        for c in MAIN_COUNTRIES:
            mask = (load_country == c)
            hourly_df[f"demand_{c}_MW"] = (
                load_p.loc[:, mask].sum(axis=1).values if mask.any() else 0.0
            )

        # ----------------------------------------------------
        # GENERATION BY COUNTRY
        # ----------------------------------------------------
        for c in MAIN_COUNTRIES:
            mask = (gen_country == c)
            hourly_df[f"generation_{c}_MW"] = (
                gen_p.loc[:, mask].sum(axis=1).values if mask.any() else 0.0
            )

        # ----------------------------------------------------
        # PRICE BY COUNTRY (LOAD-WEIGHTED)
        # ----------------------------------------------------
        prices_bus = n.buses_t.marginal_price.loc[idx]
        load_by_bus = load_p.T.groupby(n.loads.bus).sum().T

        for c in MAIN_COUNTRIES:
            buses_c = bus_country[bus_country == c].index
            buses_c = buses_c.intersection(load_by_bus.columns)

            if len(buses_c) == 0:
                hourly_df[f"price_{c}_EUR_per_MWh"] = np.nan
                continue

            load_c  = load_by_bus[buses_c]
            price_c = prices_bus[buses_c]

            denom = load_c.sum(axis=1).replace(0, np.nan)
            hourly_df[f"price_{c}_EUR_per_MWh"] = (
                (price_c * load_c).sum(axis=1) / denom
            ).values

        # ----------------------------------------------------
        # LOAD SHEDDING (EU)
        # ----------------------------------------------------
        if (n.generators.carrier == "load_shedding").any():
            hourly_df["load_shed_MW"] = (
                gen_p.loc[:, n.generators.carrier == "load_shedding"]
                .sum(axis=1)
                .values
            )
        else:
            hourly_df["load_shed_MW"] = 0.0

        # ----------------------------------------------------
        # GENERATION BY CARRIER (EU)
        # ----------------------------------------------------
        gen_by_carrier = gen_p.T.groupby(n.generators.carrier).sum().T
        for car in gen_by_carrier.columns:
            hourly_df[f"gen_{car}_MW"] = gen_by_carrier[car].values

        # ----------------------------------------------------
        # ELECTRIC EXPORT FROM ISLAND (HVDC ONLY)
        # ----------------------------------------------------
        elec_export_MW = (
            n.links_t.p0.loc[idx, n.links.carrier == "DC"].sum(axis=1)
            if (n.links.carrier == "DC").any()
            else pd.Series(0.0, index=idx)
        )

        hourly_df["elec_export_MW"] = elec_export_MW.values

        rows_hourly.append(hourly_df)

        # ----------------------------------------------------
        # DAILY METRICS
        # ----------------------------------------------------
        rows_daily.append({
            "date": d.date(),
            "demand_MWh": total_demand_MW.sum(),
            "generation_MWh": total_generation_MW.sum(),
            "avg_price_all_EUR_per_MWh": prices.stack().mean(),
            "load_shed_MWh": hourly_df["load_shed_MW"].sum(),
            "elec_export_MWh": elec_export_MW.sum(),
        })

    # --------------------------------------------------------
    # SAVE MONTH
    # --------------------------------------------------------
    out_dir = out_root / month
    out_dir.mkdir(parents=True, exist_ok=True)

    pd.DataFrame(rows_daily).set_index("date").to_csv(
        out_dir / f"daily_metrics_{month}.csv"
    )

    if rows_hourly:
        pd.concat(rows_hourly, ignore_index=True).to_csv(
            out_dir / f"hourly_power_{month}.csv",
            index=False
        )

    print(f"✓ Saved {month}")

print("\n=== FINAL M3_EI01 COMPLETED SUCCESSFULLY ===")


=== FINAL M3_EI01 — COUNTRY-CORRECT (EXPORT HUB) ===
Network: C:\Users\franc\pypsa\pypsa-eur\results\networks\EI01_export_hub.nc

=== Month 2013-01 ===


INFO:pypsa.io:Imported network EI01_export_hub.nc has buses, carriers, generators, lines, links, loads, storage_units, stores


→ 2013-01-01 (24 h)


INFO:linopy.model: Solve problem using Cbc solver
INFO:linopy.model:Solver options:
 - threads: 4
INFO:linopy.io: Writing time: 0.54s
INFO:linopy.solvers:Welcome to the CBC MILP Solver 
Version: 2.10.12 
Build Date: Aug  2 2025 

command line - cbc -printingOptions all -import C:\Users\franc\AppData\Local\Temp\linopy-problem-ho1_zj5w.lp -threads 4 -solve -solu C:\Users\franc\AppData\Local\Temp\linopy-solve-ckfsypyw.sol (default strategy 1)
Option for printingOptions changed from normal to all
No match for threads - ? for list of commands
No match for 4 - ? for list of commands
Presolve 7252 (-62592) rows, 16689 (-17035) columns and 33827 (-84838) elements
Perturbing problem by 0.001% of 763.73472 - largest nonzero change 0.00086369541 ( 0.10014579%) - largest zero change 0.00086368374
0  Obj -173.78136 Primal inf 16214754 (2056)
220  Obj -173.78136 Primal inf 18329397 (2030)
440  Obj 814046.01 Primal inf 21802834 (2114)
660  Obj 980815.31 Primal inf 18307657 (2221)
880  Obj 1182622.2 P

→ 2013-01-02 (24 h)


...

→ 2013-12-30 (24 h)


INFO:linopy.model: Solve problem using Cbc solver
INFO:linopy.model:Solver options:
 - threads: 4
INFO:linopy.io: Writing time: 0.6s
INFO:linopy.solvers:Welcome to the CBC MILP Solver 
Version: 2.10.12 
Build Date: Aug  2 2025 

command line - cbc -printingOptions all -import C:\Users\franc\AppData\Local\Temp\linopy-problem-lrbq9d_o.lp -threads 4 -solve -solu C:\Users\franc\AppData\Local\Temp\linopy-solve-d4xk08ps.sol (default strategy 1)
Option for printingOptions changed from normal to all
No match for threads - ? for list of commands
No match for 4 - ? for list of commands
Presolve 7254 (-62590) rows, 16676 (-17048) columns and 33814 (-84851) elements
Perturbing problem by 0.001% of 763.73472 - largest nonzero change 0.00086369541 ( 0.10749587%) - largest zero change 0.00086368374
0  Obj 7600071 Primal inf 16678876 (2065) Dual inf 19486.832 (2)
220  Obj 250453.27 Primal inf 19837839 (2030)
440  Obj 1592340.7 Primal inf 19893403 (2123)
660  Obj 1758325.8 Primal inf 21895001 (2218)
88

→ 2013-12-31 (24 h)


INFO:linopy.model: Solve problem using Cbc solver
INFO:linopy.model:Solver options:
 - threads: 4
INFO:linopy.io: Writing time: 0.54s
INFO:linopy.solvers:Welcome to the CBC MILP Solver 
Version: 2.10.12 
Build Date: Aug  2 2025 

command line - cbc -printingOptions all -import C:\Users\franc\AppData\Local\Temp\linopy-problem-u185sipb.lp -threads 4 -solve -solu C:\Users\franc\AppData\Local\Temp\linopy-solve-awyi_ekt.sol (default strategy 1)
Option for printingOptions changed from normal to all
No match for threads - ? for list of commands
No match for 4 - ? for list of commands
Presolve 7260 (-62584) rows, 16850 (-16874) columns and 34000 (-84665) elements
Perturbing problem by 0.001% of 763.73472 - largest nonzero change 0.00086343499 ( 0.10351277%) - largest zero change 0.00086266869
0  Obj -174.49068 Primal inf 16566393 (2072)
220  Obj -174.49068 Primal inf 19239691 (2034)
440  Obj 685674.95 Primal inf 18783493 (2110)
660  Obj 700749.17 Primal inf 18563036 (2197)
880  Obj 707449.64 P

✓ Saved 2013-12

=== FINAL M3_EI01 COMPLETED SUCCESSFULLY ===


In [5]:
# ============================================================
# FINAL M4_EI01 — YEAR AGGREGATION (2013)
# EXPORT HUB SCENARIO (ELECTRICITY ONLY)
# FULL DROP-IN — BASELINE ALIGNED
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
root = Path(r"C:\Users\franc\pypsa\pypsa-eur")

out_root = root / "results" / "daily_runs_EI01"
ei01_dir = out_root / "EI01_2013"
ei01_dir.mkdir(parents=True, exist_ok=True)

print("M4_EI01 writing to:", ei01_dir.resolve())

# ------------------------------------------------------------
# LOAD DAILY + HOURLY FILES
# ------------------------------------------------------------
months = [f"2013-{m:02d}" for m in range(1, 13)]

daily_frames = []
hourly_frames = []

for mm in months:
    dfn = out_root / mm / f"daily_metrics_{mm}.csv"
    hfn = out_root / mm / f"hourly_power_{mm}.csv"

    if dfn.exists():
        daily_frames.append(
            pd.read_csv(dfn, parse_dates=["date"]).set_index("date")
        )
    else:
        print(f"[WARN] Missing daily file: {dfn}")

    if hfn.exists():
        hourly_frames.append(
            pd.read_csv(hfn, parse_dates=["snapshot", "date"])
        )
    else:
        print(f"[WARN] Missing hourly file: {hfn}")

if not daily_frames or not hourly_frames:
    raise RuntimeError("Missing daily or hourly data — run M3_EI01 first.")

daily = pd.concat(daily_frames).sort_index()
hourly = pd.concat(hourly_frames, ignore_index=True)

print(f"Loaded {len(daily)} daily rows.")
print(f"Loaded {len(hourly)} hourly rows.")

# ============================================================
# ANNUAL KPIs (EU — BASELINE IDENTICAL)
# ============================================================

demand_total = daily["demand_MWh"].sum()
generation_total = daily["generation_MWh"].sum()

variable_cost_total = (
    daily["variable_cost_EUR"].sum()
    if "variable_cost_EUR" in daily.columns else 0.0
)

avg_price_mean = (
    daily["avg_price_all_EUR_per_MWh"].mean()
    if "avg_price_all_EUR_per_MWh" in daily.columns else np.nan
)

total_load_shed = (
    daily["load_shed_MWh"].sum()
    if "load_shed_MWh" in daily.columns else 0.0
)

days_with_shedding = (
    int((daily["load_shed_MWh"] > 0).sum())
    if "load_shed_MWh" in daily.columns else 0
)

# ============================================================
# COUNTRIES MAIN — BASELINE-ALIGNED
# ============================================================

MAIN_COUNTRIES = ["DK", "DE", "NL", "NO", "SE", "BE", "FR", "PL", "PT"]
rows = []

# ----------------------------------------
# Identify load shedding hours (EU)
# ----------------------------------------
if "load_shed_MW" in hourly.columns:
    shed_ts = hourly["load_shed_MW"]
    shedding_hours = shed_ts[shed_ts > 0].index
else:
    shedding_hours = []

# ----------------------------------------
# Precompute demand during shedding hours
# ----------------------------------------
demand_during_shed = {}
total_demand_shed = 0.0

for c in MAIN_COUNTRIES:
    dcol = f"demand_{c}_MW"
    if dcol in hourly.columns and len(shedding_hours) > 0:
        val = hourly.loc[shedding_hours, dcol].sum()
    else:
        val = 0.0
    demand_during_shed[c] = val
    total_demand_shed += val

# ----------------------------------------
# Build table (IDENTICAL to baseline)
# ----------------------------------------
for c in MAIN_COUNTRIES + ["EU"]:

    if c == "EU":
        demand = demand_total
        generation = generation_total
        price = avg_price_mean
        load_shed = total_load_shed

    else:
        dcol = f"demand_{c}_MW"
        gcol = f"generation_{c}_MW"
        pcol = f"price_{c}_EUR_per_MWh"

        demand = hourly[dcol].sum() if dcol in hourly.columns else 0.0
        generation = hourly[gcol].sum() if gcol in hourly.columns else 0.0

        # Load-weighted price
        if dcol in hourly.columns and pcol in hourly.columns:
            w = hourly[dcol].replace(0, np.nan)
            mask = w > 1e-6
            price = (hourly.loc[mask, pcol] * w.loc[mask]).sum() / w.loc[mask].sum()
        else:
            price = np.nan

        # Load shedding allocation
        if total_demand_shed > 0:
            load_shed = total_load_shed * demand_during_shed[c] / total_demand_shed
        else:
            load_shed = 0.0

    rows.append({
        "country": c,
        "annual_demand_MWh": demand,
        "annual_generation_MWh": generation,
        "avg_price_EUR_per_MWh": price,
        "load_shed_MWh": load_shed,
    })

countries_main = pd.DataFrame(rows).set_index("country")

countries_main_out = ei01_dir / "EI01_2013_countries_main.csv"
countries_main.to_csv(countries_main_out)

print(" -", countries_main_out.name)

# ============================================================
# GENERATION MIX (EU — BASELINE)
# ============================================================

gen_cols = [c for c in hourly.columns if c.startswith("gen_") and c.endswith("_MW")]

gen_by_carrier = (
    hourly[gen_cols]
    .sum()
    .rename(lambda c: c.replace("gen_", "").replace("_MW", ""))
    .drop([c for c in gen_cols if "load_shedding" in c], errors="ignore")
)

gen_mix_table = pd.DataFrame({
    "MWh": gen_by_carrier,
    "%_of_served_generation": 100.0 * gen_by_carrier / gen_by_carrier.sum()
})

# ============================================================
# CURTAILMENT & NET EXPORTS (BASELINE PLACEHOLDERS)
# ============================================================

curtailment_by_carrier = pd.Series(dtype=float, name="MWh")
net_exports_annual     = pd.Series(dtype=float, name="MWh")

# ============================================================
# SUMMARY TABLE (EU)
# ============================================================

annual_summary_df = pd.DataFrame([{
    "demand_MWh_total": demand_total,
    "generation_MWh_total": generation_total,
    "variable_cost_EUR_total": variable_cost_total,
    "avg_price_EUR_per_MWh_mean": avg_price_mean,
    "total_load_shed_MWh": total_load_shed,
    "days_with_any_load_shed": days_with_shedding,
}])

# ------------------------------------------------------------
# SAVE FILES
# ------------------------------------------------------------
daily_out   = ei01_dir / "EI01_2013_daily.csv"
summary_out = ei01_dir / "EI01_2013_summary.csv"
genmix_out  = ei01_dir / "EI01_2013_generation_mix.csv"
curt_out    = ei01_dir / "EI01_2013_curtailment.csv"
netexp_out  = ei01_dir / "EI01_2013_net_exports.csv"

daily.to_csv(daily_out)
annual_summary_df.to_csv(summary_out, index=False)
gen_mix_table.to_csv(genmix_out)
curtailment_by_carrier.to_csv(curt_out)
net_exports_annual.to_csv(netexp_out)

# ============================================================
# FINAL REPORT
# ============================================================

written_files = [
    daily_out,
    summary_out,
    genmix_out,
    curt_out,
    netexp_out,
    countries_main_out,
]

print("\n✔ EI01 M4 COMPLETE — FILES WRITTEN / UPDATED:")
for f in written_files:
    print(" -", f.name)

print("\n=== EI01 M4 COMPLETED SUCCESSFULLY ===")


M4_EI01 writing to: C:\Users\franc\pypsa\pypsa-eur\results\daily_runs_EI01\EI01_2013
Loaded 365 daily rows.
Loaded 8760 hourly rows.
 - EI01_2013_countries_main.csv

✔ EI01 M4 COMPLETE — FILES WRITTEN / UPDATED:
 - EI01_2013_daily.csv
 - EI01_2013_summary.csv
 - EI01_2013_generation_mix.csv
 - EI01_2013_curtailment.csv
 - EI01_2013_net_exports.csv
 - EI01_2013_countries_main.csv

=== EI01 M4 COMPLETED SUCCESSFULLY ===


In [6]:
# ============================================================
# FINAL M5_EI01 — CALENDAR 8 CATEGORIES (2013)
# EXPORT HUB SCENARIO (EI01)
# BASELINE-ALIGNED — DEFINITIVE VERSION
#
# CREATES:
#   - labels_by_day.csv
#   - category_summary.csv
#   - EI01_2013_country_categories.csv
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import date

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
SCENARIO = "EI01"
root = Path(r"C:\Users\franc\pypsa\pypsa-eur")

out_root = root / "results" / f"daily_runs_{SCENARIO}"
assert out_root.exists(), f"OUTDIR not found: {out_root}"

clusters_dir = out_root / "clusters" / "calendar_8cats_2013"
clusters_dir.mkdir(parents=True, exist_ok=True)

ei01_dir = out_root / "EI01_2013"
ei01_dir.mkdir(parents=True, exist_ok=True)

written_files = []

print("M5_EI01 writing to:", clusters_dir.resolve())

# ------------------------------------------------------------
# LOAD HOURLY FILES (ALL MONTHS)
# ------------------------------------------------------------
frames = []

for m in range(1, 13):
    mm = f"2013-{m:02d}"
    fn = out_root / mm / f"hourly_power_{mm}.csv"
    if not fn.exists():
        raise FileNotFoundError(fn)

    df = pd.read_csv(fn, parse_dates=["snapshot", "date"])
    df["hour"] = df["snapshot"].dt.hour
    frames.append(df)

hourly = pd.concat(frames, ignore_index=True)
print(f"Loaded {len(hourly)} hourly rows")

# ------------------------------------------------------------
# SEASON FUNCTION (IDENTICAL TO BASELINE)
# ------------------------------------------------------------
def season_2013(d):
    if date(2013,1,1) <= d <= date(2013,3,19):
        return "Winter"
    if date(2013,3,20) <= d <= date(2013,6,20):
        return "Spring"
    if date(2013,6,21) <= d <= date(2013,9,21):
        return "Summer"
    if date(2013,9,22) <= d <= date(2013,12,20):
        return "Autumn"
    return "Winter"

# ------------------------------------------------------------
# LABELS BY DAY
# ------------------------------------------------------------
labels = []

for d in sorted(hourly["date"].unique()):
    d_date = pd.Timestamp(d).date()
    season = season_2013(d_date)
    wwe = "weekend" if pd.Timestamp(d).weekday() >= 5 else "weekday"

    labels.append({
        "date": d,
        "season": season,
        "weekday_or_weekend": wwe,
        "category": f"{season.lower()}_{wwe}",
    })

labels_df = pd.DataFrame(labels).set_index("date")

f = clusters_dir / "labels_by_day.csv"
labels_df.to_csv(f)
written_files.append(f.name)

hourly = hourly.merge(labels_df, left_on="date", right_index=True, how="left")

# ------------------------------------------------------------
# DETECT VRE (STRICT — BASELINE DEFINITION)
# ------------------------------------------------------------
vre_carriers = [
    "onwind",
    "offwind-ac",
    "offwind-dc",
    "offwind-float",
    "solar",
    "solar-hsat",
    "ror",
]

vre_cols = [
    c for c in hourly.columns
    if c.startswith("gen_")
    and c.endswith("_MW")
    and any(c == f"gen_{vc}_MW" for vc in vre_carriers)
]

if not vre_cols:
    raise RuntimeError("No VRE generation columns detected")

hourly["vre_total_MW"] = hourly[vre_cols].sum(axis=1)

# ------------------------------------------------------------
# COUNTRY × CATEGORY SUMMARY (CORE OUTPUT)
# ------------------------------------------------------------
MAIN_COUNTRIES = ["DK", "DE", "NL", "NO", "SE", "BE", "FR", "PL", "PT"]
rows = []

for cat, days in labels_df.groupby("category").groups.items():
    hsub = hourly[hourly["date"].isin(days)]

    total_demand_eu = hsub["total_demand_MW"].mean()

    for c in MAIN_COUNTRIES + ["EU"]:

        if c == "EU":
            demand = total_demand_eu
            vre = hsub["vre_total_MW"].mean()

            # Demand-weighted EU price (IDENTICAL to baseline)
            num = 0.0
            den = 0.0
            for cc in MAIN_COUNTRIES:
                dcol = f"demand_{cc}_MW"
                pcol = f"price_{cc}_EUR_per_MWh"
                if dcol in hsub.columns and pcol in hsub.columns:
                    w = hsub[dcol].mean()
                    num += w * hsub[pcol].mean()
                    den += w
            price = num / den if den > 0 else np.nan

        else:
            dcol = f"demand_{c}_MW"
            pcol = f"price_{c}_EUR_per_MWh"

            demand = hsub[dcol].mean() if dcol in hsub.columns else 0.0
            share = demand / total_demand_eu if total_demand_eu > 0 else 0.0
            vre = hsub["vre_total_MW"].mean() * share

            price = hsub[pcol].mean() if pcol in hsub.columns else np.nan

        rows.append({
            "country": c,
            "category": cat,
            "season": labels_df.loc[days[0], "season"],
            "weekday_or_weekend": labels_df.loc[days[0], "weekday_or_weekend"],
            "n_days": len(days),
            "mean_daily_demand_MW": demand,
            "mean_daily_vre_MW": vre,
            "mean_price_EUR_per_MWh": price,
        })

country_categories = pd.DataFrame(rows)

f = ei01_dir / "EI01_2013_country_categories.csv"
country_categories.to_csv(f, index=False)
written_files.append(f.name)

# ------------------------------------------------------------
# CATEGORY SUMMARY (EU-LEVEL)
# ------------------------------------------------------------
summary_rows = []

for cat, days in labels_df.groupby("category").groups.items():
    hsub = hourly[hourly["date"].isin(days)]

    summary_rows.append({
        "category": cat,
        "season": labels_df.loc[days[0], "season"],
        "weekday_or_weekend": labels_df.loc[days[0], "weekday_or_weekend"],
        "n_days": len(days),
        "mean_price_EUR_per_MWh": hsub["total_demand_MW"].mean(),
    })

f = clusters_dir / "category_summary.csv"
pd.DataFrame(summary_rows).set_index("category").to_csv(f)
written_files.append(f.name)

# ------------------------------------------------------------
# FINAL REPORT
# ------------------------------------------------------------
print("\n✔ EI01 M5 COMPLETE — FILES WRITTEN:")
for fn in written_files:
    print(" -", fn)

print("\n=== EI01 M5 COMPLETED SUCCESSFULLY ===")


M5_EI01 writing to: C:\Users\franc\pypsa\pypsa-eur\results\daily_runs_EI01\clusters\calendar_8cats_2013
Loaded 8760 hourly rows

✔ EI01 M5 COMPLETE — FILES WRITTEN:
 - labels_by_day.csv
 - EI01_2013_country_categories.csv
 - category_summary.csv

=== EI01 M5 COMPLETED SUCCESSFULLY ===


In [7]:
# ============================================================
# FINAL M6_EI01 — REBUILD VRE × CATEGORY (2013)
# EXPORT HUB SCENARIO (EI01)
#
# SOURCE OF TRUTH:
#   - Hourly dispatch → gen_*_MW (from M3_EI01)
#   - Category labels → labels_by_day.csv (from M5_EI01)
#
# OUTPUT:
#   - EI01_2013_vre_by_category.csv
# ============================================================

import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
SCENARIO = "EI01"
root = Path(r"C:\Users\franc\pypsa\pypsa-eur")

out_root = root / "results" / f"daily_runs_{SCENARIO}"

clusters_dir = out_root / "clusters" / "calendar_8cats_2013"
ei01_dir = out_root / "EI01_2013"
ei01_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# LOAD CATEGORY LABELS
# ------------------------------------------------------------
labels = (
    pd.read_csv(
        clusters_dir / "labels_by_day.csv",
        parse_dates=["date"]
    )
    .set_index("date")
)

# ------------------------------------------------------------
# LOAD ALL HOURLY FILES
# ------------------------------------------------------------
hourly_frames = []

for m in range(1, 13):
    mm = f"2013-{m:02d}"
    fn = out_root / mm / f"hourly_power_{mm}.csv"

    if not fn.exists():
        print(f"[WARN] Missing hourly file: {fn}")
        continue

    df = pd.read_csv(fn, parse_dates=["snapshot"])
    df["date"] = df["snapshot"].dt.normalize()
    hourly_frames.append(df)

if not hourly_frames:
    raise RuntimeError("No hourly files found for EI01 scenario.")

hourly = pd.concat(hourly_frames, ignore_index=True)
print(f"Loaded {len(hourly)} hourly rows.")

# ------------------------------------------------------------
# DETECT VRE GENERATION COLUMNS (BASELINE-DEFINITION)
# ------------------------------------------------------------
vre_carriers = [
    "onwind",
    "offwind-ac",
    "offwind-dc",
    "offwind-float",
    "solar",
    "solar-hsat",
    "ror",
]

vre_cols = [
    c for c in hourly.columns
    if c.startswith("gen_")
    and c.endswith("_MW")
    and any(c == f"gen_{vc}_MW" for vc in vre_carriers)
]

if not vre_cols:
    raise RuntimeError(
        "No VRE generation columns found in hourly data.\n"
        f"Available columns: {list(hourly.columns)}"
    )

print("Detected VRE technologies:", vre_cols)

# ------------------------------------------------------------
# MERGE CATEGORY LABELS
# ------------------------------------------------------------
hourly = hourly.merge(
    labels[["category", "season", "weekday_or_weekend"]],
    left_on="date",
    right_index=True,
    how="inner"
)

# ============================================================
# BUILD VRE × TECHNOLOGY × CATEGORY (IDENTICAL TO BASELINE)
# ============================================================
rows = []

for col in vre_cols:
    tech = col.replace("gen_", "").replace("_MW", "")

    for cat, g in hourly.groupby("category"):
        rows.append({
            "technology": tech,
            "category": cat,
            "season": g["season"].iloc[0],
            "weekday_or_weekend": g["weekday_or_weekend"].iloc[0],
            "n_hours": len(g),
            "mean_vre_MW": g[col].mean(),
            "mean_daily_vre_MWh": g[col].mean() * 24.0,
        })

vre_by_category = (
    pd.DataFrame(rows)
    .set_index(["technology", "category"])
    .sort_index()
)

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------
out_fn = ei01_dir / "EI01_2013_vre_by_category.csv"
vre_by_category.to_csv(out_fn)

# ------------------------------------------------------------
# FINAL REPORT
# ------------------------------------------------------------
print("\n✔ EI01 M6 COMPLETE — FILE WRITTEN:")
print(" -", out_fn.name)

print("\nPreview:")
print(vre_by_category.head(12))

print("\n=== EI01 M6 COMPLETED SUCCESSFULLY ===")


Loaded 8760 hourly rows.
Detected VRE technologies: ['gen_offwind-ac_MW', 'gen_offwind-dc_MW', 'gen_offwind-float_MW', 'gen_onwind_MW', 'gen_ror_MW', 'gen_solar_MW', 'gen_solar-hsat_MW']

✔ EI01 M6 COMPLETE — FILE WRITTEN:
 - EI01_2013_vre_by_category.csv

Preview:
                           season weekday_or_weekend  n_hours   mean_vre_MW  \
technology category                                                           
offwind-ac autumn_weekday  Autumn            weekday     1560  37372.875081   
           autumn_weekend  Autumn            weekend      600  37987.601350   
           spring_weekday  Spring            weekday     1608  32296.770490   
           spring_weekend  Spring            weekend      624  30206.071317   
           summer_weekday  Summer            weekday     1584  23712.428227   
           summer_weekend  Summer            weekend      648  29447.504111   
           winter_weekday  Winter            weekday     1512  35326.034579   
           winter_weeke

In [1]:
# ============================================================
# FINAL M7 — ECONOMIC ACCOUNTING BY CATEGORY (2013)
# GENERIC: EI01 (Export Hub) + EI03 (Multi-vector)
#
# SOURCES:
#   - Hourly dispatch (M3)
#   - Calendar categories (M5)
#
# OUTPUTS:
#   - *_2013_costs_by_category.csv
#   - *_2013_annual_costs.csv
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
SCENARIO = "EI01"   # or "EI03"

root = Path(r"C:\Users\franc\pypsa\pypsa-eur")
out_root = root / "results" / f"daily_runs_{SCENARIO}"

clusters_dir = out_root / "clusters" / "calendar_8cats_2013"
year_dir = out_root / f"{SCENARIO}_2013"
year_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# ECONOMIC PARAMETERS
# ------------------------------------------------------------
COSTS = {
    "offwind": 0.0,
    "hvdc": 1.0,
    "electrolysis": 60.0,      # €/MWh_el
    "h2_storage": 0.1,         # €/MWh_H2
    "curtailment": 5.0,        # €/MWh
    "load_shedding": 10_000.0,
}

H2_PRICE = 90.0   # €/MWh_H2

# ------------------------------------------------------------
# LOAD CATEGORY LABELS
# ------------------------------------------------------------
labels = (
    pd.read_csv(
        clusters_dir / "labels_by_day.csv",
        parse_dates=["date"]
    )
    .set_index("date")
)

# ------------------------------------------------------------
# LOAD ALL HOURLY FILES
# ------------------------------------------------------------
frames = []

for m in range(1, 13):
    mm = f"2013-{m:02d}"
    fn = out_root / mm / f"hourly_power_{mm}.csv"
    if not fn.exists():
        raise FileNotFoundError(fn)

    df = pd.read_csv(fn, parse_dates=["snapshot"])
    df["date"] = df["snapshot"].dt.normalize()
    frames.append(df)

hourly = pd.concat(frames, ignore_index=True)
print(f"Loaded {len(hourly)} hourly rows")

# ------------------------------------------------------------
# MERGE CATEGORIES
# ------------------------------------------------------------
hourly = hourly.merge(
    labels[["category", "season", "weekday_or_weekend"]],
    left_on="date",
    right_index=True,
    how="inner"
)

# ------------------------------------------------------------
# DETECT SCENARIO FEATURES
# ------------------------------------------------------------
has_h2 = {"elec_to_h2_MW", "h2_produced_MW"}.issubset(hourly.columns)
has_export = "elec_export_MW" in hourly.columns

print("H2 active:", has_h2)
print("Electric export active:", has_export)

# ------------------------------------------------------------
# DETECT OFFSHORE WIND
# ------------------------------------------------------------
offwind_cols = [
    c for c in hourly.columns
    if c.startswith("gen_") and "offwind" in c
]

if not offwind_cols:
    raise RuntimeError("No offshore wind generation detected")

# ------------------------------------------------------------
# DETECT PRICES BY COUNTRY
# ------------------------------------------------------------
price_cols = {
    c.replace("price_", "").replace("_EUR_per_MWh", ""): c
    for c in hourly.columns
    if c.startswith("price_") and c.endswith("_EUR_per_MWh")
}

# ------------------------------------------------------------
# MAIN AGGREGATION
# ------------------------------------------------------------
rows = []

for cat, g in hourly.groupby("category"):

    out = {
        "category": cat,
        "season": g["season"].iloc[0],
        "weekday_or_weekend": g["weekday_or_weekend"].iloc[0],
        "n_hours": len(g),
    }

    # -----------------------------
    # Offshore wind
    # -----------------------------
    offwind_MWh = g[offwind_cols].sum(axis=1).sum()
    out["offwind_MWh"] = offwind_MWh
    out["cost_offwind_EUR"] = offwind_MWh * COSTS["offwind"]

    # -----------------------------
    # Curtailment (proxy)
    # -----------------------------
    if "p_max_pu" in g.columns:
        # optional if ever exported
        curtailment_MWh = g["p_max_pu"].sum() - offwind_MWh
    else:
        curtailment_MWh = 0.0

    out["curtailment_MWh"] = max(curtailment_MWh, 0.0)
    out["cost_curtailment_EUR"] = out["curtailment_MWh"] * COSTS["curtailment"]

    # -----------------------------
    # Load shedding
    # -----------------------------
    if "load_shed_MW" in g.columns:
        shed_MWh = g["load_shed_MW"].sum()
    else:
        shed_MWh = 0.0

    out["load_shed_MWh"] = shed_MWh
    out["cost_load_shedding_EUR"] = shed_MWh * COSTS["load_shedding"]

    # -----------------------------
    # Electric export revenue
    # -----------------------------
    revenue_elec = 0.0

    if has_export:
        for country, pcol in price_cols.items():
            # proportional allocation by price country
            revenue_elec += (
                g["elec_export_MW"] * g[pcol]
            ).sum()

    out["revenue_electricity_EUR"] = revenue_elec

    # -----------------------------
    # H2 economics (EU only)
    # -----------------------------
    if has_h2:
        h2_MWh = g["h2_produced_MW"].sum()
        elec_to_h2_MWh = g["elec_to_h2_MW"].sum()

        out["h2_produced_MWh"] = h2_MWh
        out["cost_electrolysis_EUR"] = elec_to_h2_MWh * COSTS["electrolysis"]
        out["revenue_h2_EUR"] = h2_MWh * H2_PRICE
    else:
        out["h2_produced_MWh"] = 0.0
        out["cost_electrolysis_EUR"] = 0.0
        out["revenue_h2_EUR"] = 0.0

    # -----------------------------
    # Net system value
    # -----------------------------
    out["net_system_value_EUR"] = (
        out["revenue_electricity_EUR"]
        + out["revenue_h2_EUR"]
        - out["cost_offwind_EUR"]
        - out["cost_electrolysis_EUR"]
        - out["cost_curtailment_EUR"]
        - out["cost_load_shedding_EUR"]
    )

    rows.append(out)

# ------------------------------------------------------------
# SAVE OUTPUTS
# ------------------------------------------------------------
df_cat = pd.DataFrame(rows).set_index("category").sort_index()

out_cat = year_dir / f"{SCENARIO}_2013_costs_by_category.csv"
df_cat.to_csv(out_cat)

df_annual = df_cat.sum(numeric_only=True).to_frame(name="value")
out_annual = year_dir / f"{SCENARIO}_2013_annual_costs.csv"
df_annual.to_csv(out_annual)

# ------------------------------------------------------------
# FINAL REPORT
# ------------------------------------------------------------
print("\n✔ M7 COMPLETE — FILES WRITTEN:")
print(" -", out_cat.name)
print(" -", out_annual.name)

print("\n=== M7 COMPLETED SUCCESSFULLY ===")


Loaded 8760 hourly rows
H2 active: False
Electric export active: True

✔ M7 COMPLETE — FILES WRITTEN:
 - EI01_2013_costs_by_category.csv
 - EI01_2013_annual_costs.csv

=== M7 COMPLETED SUCCESSFULLY ===


In [1]:
# ============================================================
# FINAL M8_EI01 — CAPEX + SYSTEM VALUE (2013)
# EXPORT HUB — ELECTRICITY ONLY
# ============================================================

import pypsa
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
SCENARIO = "EI01"
YEAR = 2013

root = Path(r"C:\Users\franc\pypsa\pypsa-eur")

network_fn = root / "results" / "networks" / "EI01_export_hub.nc"
out_root   = root / "results" / f"daily_runs_{SCENARIO}"
scenario_dir = out_root / f"{SCENARIO}_{YEAR}"
scenario_dir.mkdir(parents=True, exist_ok=True)

print("\nM8 — loading network:")
print(" ", network_fn.name)

assert network_fn.exists(), f"Network not found: {network_fn}"

n = pypsa.Network(network_fn)

# ------------------------------------------------------------
# LOAD ANNUAL ECONOMIC RESULTS FROM M7
# ------------------------------------------------------------
annual_costs_fn = scenario_dir / f"{SCENARIO}_{YEAR}_annual_costs.csv"
assert annual_costs_fn.exists(), "Run M7_EI01 before M8."

annual_costs = pd.read_csv(annual_costs_fn)

# robust handling
if "metric" in annual_costs.columns:
    annual_costs = annual_costs.set_index("metric")
else:
    annual_costs = annual_costs.set_index(annual_costs.columns[0])

net_system_value_before_capex = float(
    annual_costs.loc["net_system_value_EUR", "value"]
)

# ------------------------------------------------------------
# CAPEX ASSUMPTIONS (€/unit) — ELECTRICITY ONLY
# ------------------------------------------------------------
CAPEX = {
    "offwind_EUR_per_MW":  2_400_000,
    "hvdc_EUR_per_MW":       600_000,
    "battery_EUR_per_MWh":   400_000,
}

LIFETIME_YEARS = {
    "offwind": 25,
    "hvdc":    40,
    "battery": 15,
}

# ------------------------------------------------------------
# INSTALLED CAPACITIES
# ------------------------------------------------------------
offwind_MW = n.generators.loc[
    n.generators.carrier.str.contains("offwind"),
    "p_nom"
].sum()

hvdc_MW = n.links.loc[
    n.links.carrier == "DC",
    "p_nom"
].sum()

# Battery = smoothing only (0.5 GWh expected)
battery_MWh = n.storage_units.loc[
    n.storage_units.carrier == "battery",
    "p_nom"
].sum() * n.storage_units.loc[
    n.storage_units.carrier == "battery",
    "max_hours"
].mean()

print("\nInstalled capacities:")
print(f" - Offshore wind: {offwind_MW:.1f} MW")
print(f" - HVDC:          {hvdc_MW:.1f} MW")
print(f" - Battery:       {battery_MWh:.1f} MWh")

# ------------------------------------------------------------
# ANNUALISED CAPEX
# ------------------------------------------------------------
capex_offwind_EUR = (
    offwind_MW * CAPEX["offwind_EUR_per_MW"]
    / LIFETIME_YEARS["offwind"]
)

capex_hvdc_EUR = (
    hvdc_MW * CAPEX["hvdc_EUR_per_MW"]
    / LIFETIME_YEARS["hvdc"]
)

capex_battery_EUR = (
    battery_MWh * CAPEX["battery_EUR_per_MWh"]
    / LIFETIME_YEARS["battery"]
)

total_capex_EUR = (
    capex_offwind_EUR
    + capex_hvdc_EUR
    + capex_battery_EUR
)

# ------------------------------------------------------------
# FINAL SYSTEM VALUE
# ------------------------------------------------------------
net_system_value_after_capex_EUR = (
    net_system_value_before_capex - total_capex_EUR
)

# ------------------------------------------------------------
# OUTPUT TABLE
# ------------------------------------------------------------
rows = [
    ("capex_offwind_EUR", capex_offwind_EUR),
    ("capex_hvdc_EUR", capex_hvdc_EUR),
    ("capex_battery_EUR", capex_battery_EUR),
    ("total_capex_EUR", total_capex_EUR),
    ("net_system_value_before_capex_EUR", net_system_value_before_capex),
    ("net_system_value_after_capex_EUR", net_system_value_after_capex_EUR),
]

df_out = pd.DataFrame(rows, columns=["metric", "value"]).set_index("metric")

out_fn = scenario_dir / f"{SCENARIO}_{YEAR}_capex_and_system_value.csv"
df_out.to_csv(out_fn)

# ------------------------------------------------------------
# FINAL REPORT
# ------------------------------------------------------------
print("\n✔ M8_EI01 COMPLETE — CAPEX ACCOUNTING ADDED")
print(" -", out_fn.name)

print("\nSummary:")
for k, v in df_out["value"].items():
    print(f" {k:35s}: {v:,.0f} €")



M8 — loading network:
  EI01_export_hub.nc


INFO:pypsa.io:Imported network EI01_export_hub.nc has buses, carriers, generators, lines, links, loads, storage_units, stores



Installed capacities:
 - Offshore wind: 62820.6 MW
 - HVDC:          44270.0 MW
 - Battery:       500.0 MWh

✔ M8_EI01 COMPLETE — CAPEX ACCOUNTING ADDED
 - EI01_2013_capex_and_system_value.csv

Summary:
 capex_offwind_EUR                  : 6,030,777,600 €
 capex_hvdc_EUR                     : 664,050,000 €
 capex_battery_EUR                  : 13,333,333 €
 total_capex_EUR                    : 6,708,160,933 €
 net_system_value_before_capex_EUR  : 31,248,257,701 €
 net_system_value_after_capex_EUR   : 24,540,096,768 €


In [2]:
# ============================================================
# FINAL M9 — WELFARE & PRICE IMPACT VS BASELINE (2013)
# COMPARES: Baseline vs Energy Island (EI01 or EI03)
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
SCENARIO = "EI01"   # "EI01" or "EI03"
YEAR = 2013

MAIN_COUNTRIES = ["DK", "DE", "NL", "NO", "SE", "BE", "FR", "PL", "PT"]

VOLL = 10_000.0  # €/MWh (Value of Lost Load)

# ------------------------------------------------------------
# PATHS (FIXED & ROBUST)
# ------------------------------------------------------------
root = Path(r"C:\Users\franc\pypsa\pypsa-eur")

baseline_dir = root / "results" / "daily_runs" / "baseline_nohub_2013"
scenario_dir = root / "results" / f"daily_runs_{SCENARIO}" / f"{SCENARIO}_2013"

assert baseline_dir.exists(), baseline_dir
assert scenario_dir.exists(), scenario_dir

# ------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------
baseline = pd.read_csv(
    baseline_dir / "baseline_nohub_2013_countries_main.csv",
    index_col="country"
)

scenario = pd.read_csv(
    scenario_dir / f"{SCENARIO}_2013_countries_main.csv",
    index_col="country"
)

# ------------------------------------------------------------
# SANITY CHECKS
# ------------------------------------------------------------
assert all(c in baseline.index for c in MAIN_COUNTRIES)
assert all(c in scenario.index for c in MAIN_COUNTRIES)

# ------------------------------------------------------------
# WELFARE CALCULATION
# ------------------------------------------------------------
rows = []

for c in MAIN_COUNTRIES:

    demand = baseline.loc[c, "annual_demand_MWh"]

    # -------------------------
    # PRICE EFFECT
    # -------------------------
    price_baseline = baseline.loc[c, "avg_price_EUR_per_MWh"]
    price_scenario = scenario.loc[c, "avg_price_EUR_per_MWh"]

    delta_price = price_scenario - price_baseline
    welfare_price = - delta_price * demand   # price drop = positive welfare

    # -------------------------
    # LOAD SHEDDING EFFECT
    # -------------------------
    shed_baseline = baseline.loc[c, "load_shed_MWh"]
    shed_scenario = scenario.loc[c, "load_shed_MWh"]

    delta_shed = shed_scenario - shed_baseline
    welfare_shed = - delta_shed * VOLL        # less shedding = positive welfare

    # -------------------------
    # TOTAL
    # -------------------------
    total_welfare = welfare_price + welfare_shed

    rows.append({
        "country": c,
        "delta_price_EUR_per_MWh": delta_price,
        "welfare_price_EUR": welfare_price,
        "delta_load_shed_MWh": delta_shed,
        "welfare_load_shed_EUR": welfare_shed,
        "total_welfare_EUR": total_welfare,
    })

welfare_df = pd.DataFrame(rows).set_index("country")

# ------------------------------------------------------------
# EU AGGREGATE
# ------------------------------------------------------------
welfare_df.loc["EU"] = {
    "delta_price_EUR_per_MWh": np.nan,
    "welfare_price_EUR": welfare_df["welfare_price_EUR"].sum(),
    "delta_load_shed_MWh": welfare_df["delta_load_shed_MWh"].sum(),
    "welfare_load_shed_EUR": welfare_df["welfare_load_shed_EUR"].sum(),
    "total_welfare_EUR": welfare_df["total_welfare_EUR"].sum(),
}

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------
out_fn = scenario_dir / f"{SCENARIO}_2013_welfare_vs_baseline.csv"
welfare_df.to_csv(out_fn)

print("\n✔ M9 COMPLETE — WELFARE VS BASELINE")
print("Scenario:", SCENARIO)
print("Output:", out_fn)
print("\nEU total welfare gain:")
print(f"{welfare_df.loc['EU', 'total_welfare_EUR']:,.0f} €")



✔ M9 COMPLETE — WELFARE VS BASELINE
Scenario: EI01
Output: C:\Users\franc\pypsa\pypsa-eur\results\daily_runs_EI01\EI01_2013\EI01_2013_welfare_vs_baseline.csv

EU total welfare gain:
33,596,875,078 €


In [3]:
# ============================================================
# FINAL M10_EI01 — EU CONSUMER WELFARE (ΔCS − OPEX)
# EXPORT HUB (EI01)
#
# DEFINIÇÃO ECONÓMICA:
#   Year 0  : − CAPEX
#   Year ≥1 : Δ Consumer Surplus EU − OPEX
#
# ONDE:
#   ΔCS = Σ_c (P_baseline − P_EI01) × Demand_EI01
#
# OUTPUT:
#   results/daily_runs_EI01/welfares/
#     EI01_eu_consumer_welfare.csv
# ============================================================

import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
SCENARIO = "EI01"
HORIZON_YEARS = 40

MAIN_COUNTRIES = ["DK", "DE", "NL", "NO", "SE", "BE", "FR", "PL", "PT"]

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
root = Path(r"C:\Users\franc\pypsa\pypsa-eur")

baseline_dir = root / "results" / "daily_runs" / "baseline_nohub_2013"
scenario_dir = root / "results" / "daily_runs_EI01" / "EI01_2013"

welfare_dir = root / "results" / "daily_runs_EI01" / "welfares"
welfare_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# LOAD INPUT FILES
# ------------------------------------------------------------
baseline = pd.read_csv(
    baseline_dir / "baseline_nohub_2013_countries_main.csv",
    index_col="country"
)

scenario = pd.read_csv(
    scenario_dir / "EI01_2013_countries_main.csv",
    index_col="country"
)

annual_costs = pd.read_csv(
    scenario_dir / "EI01_2013_annual_costs.csv",
    index_col=0
)

capex_df = pd.read_csv(
    scenario_dir / "EI01_2013_capex_and_system_value.csv",
    index_col=0
)

# ------------------------------------------------------------
# 1) Δ CONSUMER SURPLUS (EU)
# ------------------------------------------------------------
delta_cs_eu = 0.0

for c in MAIN_COUNTRIES:
    demand = scenario.loc[c, "annual_demand_MWh"]
    p_base = baseline.loc[c, "avg_price_EUR_per_MWh"]
    p_ei   = scenario.loc[c, "avg_price_EUR_per_MWh"]

    delta_cs_eu += (p_base - p_ei) * demand

# ------------------------------------------------------------
# 2) OPEX — EXPORT HUB ONLY
# ------------------------------------------------------------
# Apenas custos operacionais reais do EI01
OPEX_KEYS_EI01 = [
    "cost_hvdc_EUR",
    "cost_battery_EUR",
    "cost_curtailment_EUR",
]

opex = annual_costs.loc[
    annual_costs.index.intersection(OPEX_KEYS_EI01),
    "value"
].sum()

# ------------------------------------------------------------
# 3) ANNUAL WELFARE (STEADY STATE)
# ------------------------------------------------------------
annual_welfare = delta_cs_eu - opex

# ------------------------------------------------------------
# 4) CAPEX (YEAR 0)
# ------------------------------------------------------------
total_capex = capex_df.loc["total_capex_EUR", "value"]

# ------------------------------------------------------------
# 5) INTERTEMPORAL WELFARE TABLE
# ------------------------------------------------------------
rows = []

cumulative = -total_capex

rows.append({
    "year": 0,
    "annual_welfare_EUR": -total_capex,
    "cumulative_welfare_EUR": cumulative,
    "note": "CAPEX"
})

for y in range(1, HORIZON_YEARS + 1):
    cumulative += annual_welfare
    rows.append({
        "year": y,
        "annual_welfare_EUR": annual_welfare,
        "cumulative_welfare_EUR": cumulative,
        "note": ""
    })

df = pd.DataFrame(rows)

# ------------------------------------------------------------
# 6) PAYBACK YEAR
# ------------------------------------------------------------
payback = df[df["cumulative_welfare_EUR"] >= 0]
payback_year = int(payback.iloc[0]["year"]) if not payback.empty else None

# ------------------------------------------------------------
# SAVE OUTPUT
# ------------------------------------------------------------
out_fn = welfare_dir / "EI01_eu_consumer_welfare.csv"
df.to_csv(out_fn, index=False)

# ------------------------------------------------------------
# PRINT RESULTS
# ------------------------------------------------------------
print("\n=== M10 — EU CONSUMER WELFARE (ΔCS − OPEX) ===\n")
print(f"Scenario: {SCENARIO}")
print(f"Δ Consumer Surplus EU : {delta_cs_eu:,.0f} € / year")
print(f"OPEX (annual)        : {opex:,.0f} € / year")
print(f"Annual welfare       : {annual_welfare:,.0f} € / year")
print(f"CAPEX (year 0)       : {-total_capex:,.0f} €\n")

print("YEAR | Annual welfare (€) | Cumulative welfare (€)")
print("-" * 60)

for _, r in df.iterrows():
    print(
        f"{int(r.year):4d} | "
        f"{r.annual_welfare_EUR:>20,.0f} | "
        f"{r.cumulative_welfare_EUR:>26,.0f}"
    )

print("\nPayback year:", payback_year if payback_year is not None else "No payback in horizon")
print("\n✔ M10_EI01 COMPLETE — FILE WRITTEN:")
print(" -", out_fn)



=== M10 — EU CONSUMER WELFARE (ΔCS − OPEX) ===

Scenario: EI01
Δ Consumer Surplus EU : 7,823,166,156 € / year
OPEX (annual)        : 0 € / year
Annual welfare       : 7,823,166,156 € / year
CAPEX (year 0)       : -6,708,160,933 €

YEAR | Annual welfare (€) | Cumulative welfare (€)
------------------------------------------------------------
   0 |       -6,708,160,933 |             -6,708,160,933
   1 |        7,823,166,156 |              1,115,005,222
   2 |        7,823,166,156 |              8,938,171,378
   3 |        7,823,166,156 |             16,761,337,533
   4 |        7,823,166,156 |             24,584,503,689
   5 |        7,823,166,156 |             32,407,669,845
   6 |        7,823,166,156 |             40,230,836,000
   7 |        7,823,166,156 |             48,054,002,156
   8 |        7,823,166,156 |             55,877,168,311
   9 |        7,823,166,156 |             63,700,334,467
  10 |        7,823,166,156 |             71,523,500,623
  11 |        7,823,166,156 |